# CNN Model Analysis Notebook

Loads a trained CNN model from an experiment directory and runs:
1. **Per-station error analysis** – spatial error maps, bias map, rankings, isolation plot
2. **Input channel visualisation** – the three gridded input channels at a chosen timestep
3. **Grid prediction** – the CNN's native H×W output field (no IDW re-interpolation needed)

In [ ]:
import os, sys
_ROOT = os.path.dirname(os.path.abspath('.'))
if os.getcwd().endswith('rainfall_data_fusion'):
    _ROOT = os.getcwd()
else:
    os.chdir(_ROOT)
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import xarray as xr
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import HTML

from src.sampling.main import stratified_spatial_kfold_dual
from src.utils import read_config
from src.raingauge.utils import load_raingauge_dataset
from src.radar.utils import load_processed_dataset
from src.visualization.main import visualise_singapore_outline, visualise_with_basemap
from src.visualization.error_analysis import get_viz_scales

from benchmarks.models.cnn import RainfallCNN
from benchmarks.processing.gridify import make_grid_coords, precompute_idw_weights, apply_idw_weights
from benchmarks.run_cnn import sample_field_at_stations, _prepare_cml_series, _prepare_radar_series
from benchmarks.visualization import (
    compute_per_station_metrics,
    plot_spatial_error_map,
    plot_bias_map,
    plot_error_ranking,
    plot_error_vs_isolation,
    plot_density_scatter,
    plot_station_rainfall_timeseries,
    plot_rainfall_grid,
    plot_rainfall_sequence,
    animate_rainfall_grid,
)

%load_ext autoreload
%autoreload 2

## Configuration

Set the experiment name and fold index here.

In [ ]:
# ── CHANGE THESE ──────────────────────────────────────────────────────────────
EXPERIMENT_NAME = "20260607_205713_cnn"
FOLD_IDX        = 0          # 0-based: fold 1 = index 0
# ──────────────────────────────────────────────────────────────────────────────

EXPERIMENT_DIR = f"experiments/{EXPERIMENT_NAME}"
CKPT_PATH      = f"{EXPERIMENT_DIR}/fold_{FOLD_IDX + 1}_model.pt"

config     = read_config("config.yaml")
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BOUNDS     = config["dataset_parameters"]["geography_bounds"]
FOLD_COUNT = config["training_params"]["fold_count"]

_viz            = config.get("analysis", {})
TIME_START      = _viz.get("time_start", None)
TIME_END        = _viz.get("time_end",   None)
SELECTED_STATION = _viz.get("selected_station", None)

SCALES         = get_viz_scales("config.yaml")
COMPARE_METRICS = SCALES["compare_metrics"]
METRIC_SCALES  = SCALES["metric_scales"]
RAINFALL_SCALE = SCALES["rainfall"]

print(f"Experiment : {EXPERIMENT_NAME}")
print(f"Checkpoint : {CKPT_PATH}")
print(f"Device     : {device}")
print(f"Bounds     : {BOUNDS}")
print(f"Folds      : {FOLD_COUNT}")
print(f"Time window: {TIME_START} → {TIME_END}")
print(f"Compare    : {COMPARE_METRICS}")
print(f"Rainfall scale: vmin={RAINFALL_SCALE['vmin']}, vmax={RAINFALL_SCALE['vmax']}")

## Load Datasets

In [ ]:
uptime    = config["filters"]["uptime_threshold"]
start     = config["dataset_parameters"]["start_year"]
end       = config["dataset_parameters"]["end_year"]
cml_nc_path = f"database/{config['dataset_parameters']['cml_folder']}"

raingauge_df, mapping_df = load_raingauge_dataset(start=start, end=end,
                                                   uptime_threshold=uptime)
raingauge_df = raingauge_df.resample("15min", closed="left", label="left").mean()
raingauge_df = raingauge_df[raingauge_df.index.minute % 15 == 0]

radar_df = load_processed_dataset("database/processed_radar_dataset.pkl")

with xr.open_dataset(cml_nc_path, engine="netcdf4") as _ds:
    cml_ts = set(pd.to_datetime(_ds["time"].values).tolist())

print(f"Raingauge : {raingauge_df.shape}")
print(f"Radar     : {radar_df.shape}")
print(f"Mapping   : {mapping_df.shape}")

## Align Timestamps

In [ ]:
radar_ts = set(radar_df["timestamp"].tolist())
common   = sorted(raingauge_df.index.intersection(radar_ts).intersection(cml_ts))
timestamps = pd.DatetimeIndex(common)

raingauge_df = raingauge_df.loc[timestamps]
radar_df     = radar_df[radar_df["timestamp"].isin(set(timestamps))].copy()

T = len(timestamps)
print(f"Aligned timesteps: {T}")
print(f"Range: {timestamps[0]} → {timestamps[-1]}")

## Spatial Folds

In [ ]:
split_info = stratified_spatial_kfold_dual(
    mapping_df, seed=config["training_params"]["seed"],
    plot=False, n_splits=FOLD_COUNT
)

train_ids = split_info[FOLD_IDX]["ml"]["train"]
val_ids   = split_info[FOLD_IDX]["ml"]["validation"]
test_ids  = split_info[FOLD_IDX]["ml"]["test"]

def _get_coords(station_ids):
    rows = mapping_df[mapping_df["id"].isin(station_ids)].set_index("id").reindex(station_ids)
    return rows["longitude"].values, rows["latitude"].values

train_lons, train_lats = _get_coords(train_ids)
val_lons,   val_lats   = _get_coords(val_ids)
test_lons,  test_lats  = _get_coords(test_ids)

print(f"Fold {FOLD_IDX + 1}:  train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}")
print(f"Test stations: {list(test_ids)}")

## Build Input Grids

The CNN takes three gridded input channels at 0.01° resolution:
- **Ch 0** Rain gauge IDW (training stations only)
- **Ch 1** CML specific attenuation IDW
- **Ch 2** Radar reflectivity

In [ ]:
grid_lons, grid_lats = make_grid_coords()
H = len(grid_lats)
W = len(grid_lons)
print(f"Grid: H={H}  W={W}  (lats descending top→bottom, lons ascending left→right)")

# Rain gauge grid (training stations only)
print("Building rain gauge grid …")
rg_weights = precompute_idw_weights(train_lons, train_lats, grid_lons, grid_lats)
rg_values  = raingauge_df[list(train_ids)].fillna(0).values.astype(np.float32)
rg_grids   = apply_idw_weights(rg_weights, rg_values, H, W)   # [T, H, W]

# CML grid
print("Building CML grid …")
cml_grids = _prepare_cml_series(cml_nc_path, grid_lons, grid_lats, timestamps)  # [T, H, W]

# Radar grid
print("Building radar grid …")
radar_grids = _prepare_radar_series(radar_df, timestamps, H, W)  # [T, H, W]

# Stack into [T, 3, H, W]
input_np = np.stack([rg_grids, cml_grids, radar_grids], axis=1).astype(np.float32)
input_t  = torch.tensor(input_np)
print(f"Input tensor: {tuple(input_t.shape)}  (T, channels, H, W)")

## Load CNN Model

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=device)
model_cfg = ckpt["model_config"]
print(f"Model config: {model_cfg}")

model = RainfallCNN(**model_cfg).to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(model)

## Run Inference

The CNN directly outputs a full H×W rainfall field per timestep.  We also bilinearly
sample the output at test station locations for per-station metrics.

In [ ]:
# Dummy targets — only the grid input matters for inference
test_targets = torch.zeros(T, len(test_ids))
test_ds      = torch.utils.data.TensorDataset(input_t, test_targets)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

all_fields = []       # [T, H, W]  — raw CNN output field
all_pred_at = []      # [T, N_test] — sampled at test station locations

with torch.no_grad():
    for batch_grids, _ in test_loader:
        batch_grids = batch_grids.to(device)
        pred_field  = model(batch_grids).clamp(min=0.0)          # [B, 1, H, W]
        all_fields.append(pred_field[:, 0].cpu().numpy())         # [B, H, W]
        pred_at = sample_field_at_stations(
            pred_field, test_lons, test_lats, grid_lons, grid_lats
        )                                                          # [B, N_test]
        all_pred_at.append(pred_at.cpu().numpy())

pred_fields  = np.concatenate(all_fields,   axis=0)  # [T, H, W]
pred_at_test = np.concatenate(all_pred_at,  axis=0)  # [T, N_test]
test_actuals = raingauge_df[list(test_ids)].fillna(0).values.astype(np.float32)  # [T, N_test]

print(f"pred_fields  : {pred_fields.shape}   range [{pred_fields.min():.3f}, {pred_fields.max():.3f}] mm")
print(f"pred_at_test : {pred_at_test.shape}")
print(f"test_actuals : {test_actuals.shape}")

# DataFrames for time-series plots (indexed by timestamp)
predictions_df = pd.DataFrame(pred_at_test, index=timestamps, columns=list(test_ids))
actuals_df     = pd.DataFrame(test_actuals,  index=timestamps, columns=list(test_ids))

---
# Part 1 – Per-Station Error Analysis

In [ ]:
per_station_data = {
    sid: {
        "actual":    test_actuals[:, i].tolist(),
        "predicted": pred_at_test[:, i].tolist(),
    }
    for i, sid in enumerate(test_ids)
}

coordinates = {
    row["id"]: (row["latitude"], row["longitude"])
    for _, row in mapping_df.iterrows()
}

station_df = compute_per_station_metrics(per_station_data, coordinates)

print(f"Test stations analysed: {len(station_df)}")
station_df.sort_values("mae", ascending=False).head(10)

### Spatial Error Maps

In [ ]:
import os
os.makedirs(f"{EXPERIMENT_DIR}/analysis", exist_ok=True)

m1, m2 = COMPARE_METRICS
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
plot_spatial_error_map(station_df, BOUNDS, metric=m1, ax=axes[0],
                       title=f"CNN — per-station {m1.upper()}",
                       show_outline=True, **METRIC_SCALES.get(m1, {}))
plot_spatial_error_map(station_df, BOUNDS, metric=m2, ax=axes[1],
                       title=f"CNN — per-station {m2.upper()}",
                       show_outline=True, **METRIC_SCALES.get(m2, {}))
fig.suptitle(f"CNN spatial error maps — fold {FOLD_IDX + 1}", fontsize=14)
plt.tight_layout()
plt.savefig(f"{EXPERIMENT_DIR}/analysis/cnn_spatial_error_maps.png", dpi=200, bbox_inches="tight")
plt.show()

### Bias Map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
plot_bias_map(station_df, BOUNDS, top_n_labels=5, ax=ax,
              title="CNN — prediction bias per station")
visualise_singapore_outline(ax=ax)
visualise_with_basemap(ax=ax)
plt.tight_layout()
plt.savefig(f"{EXPERIMENT_DIR}/analysis/cnn_bias_map.png", dpi=200, bbox_inches="tight")
plt.show()

### Station Rankings

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
plot_error_ranking(station_df, metric="mae", top_n=15, ax=ax1,
                   title="CNN — worst stations by MAE")
plot_error_ranking(station_df, metric="f1",  top_n=15, ax=ax2,
                   title="CNN — worst stations by F1")
fig.suptitle("CNN — worst-performing stations", fontsize=13)
plt.tight_layout()
plt.savefig(f"{EXPERIMENT_DIR}/analysis/cnn_rankings.png", dpi=200, bbox_inches="tight")
plt.show()

### Error vs. Spatial Isolation

In [ ]:
train_ids_list = list(train_ids)
fig, ax = plt.subplots(figsize=(7, 5))
plot_error_vs_isolation(
    station_df, train_ids_list, coordinates,
    metric="mae", ax=ax,
    title="CNN — MAE vs. distance to nearest training gauge",
)
plt.tight_layout()
plt.savefig(f"{EXPERIMENT_DIR}/analysis/cnn_isolation.png", dpi=200, bbox_inches="tight")
plt.show()

### Density Scatter — Actual vs Predicted

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_density_scatter(
    actual=test_actuals.ravel(),
    predicted=pred_at_test.ravel(),
    model_name="CNN",
    title="CNN — Actual vs Predicted (density scatter)",
    log_scale=True,
    ax=ax,
)
plt.tight_layout()
plt.savefig(f"{EXPERIMENT_DIR}/analysis/cnn_density_scatter.png", dpi=200, bbox_inches="tight")
plt.show()

### Station Time Series

In [ ]:
# Pick which station and time window to plot
_first_test = list(test_ids)[0]
STATION_ID  = SELECTED_STATION if SELECTED_STATION in predictions_df.columns else _first_test
TS_START    = "2025-01-10"   # ← change to zoom into a specific event
TS_END      = "2025-01-12"

print(f"Plotting station: {STATION_ID}  |  {TS_START} → {TS_END}")

fig, ax = plt.subplots(figsize=(16, 4))
plot_station_rainfall_timeseries(
    predictions_df=predictions_df,
    actuals_df=actuals_df,
    station_id=STATION_ID,
    time_start=TS_START,
    time_end=TS_END,
    title=f"CNN — Station {STATION_ID}",
    ax=ax,
)
plt.tight_layout()
plt.show()

---
# Part 2 – Input Channel Visualisation

Shows the three gridded input channels at a single chosen timestep.
This is unique to the CNN — the GNN never forms an explicit spatial grid.

In [ ]:
CHANNEL_TIMESTEP = 500   # ← change to any index in [0, T)

channel_names = [
    "Ch 0 — Rain gauge IDW (train stations)",
    "Ch 1 — CML specific attenuation IDW",
    "Ch 2 — Radar reflectivity",
]
# Grid lats are descending (top→bottom); flip for imshow with origin='lower'
channels = input_np[CHANNEL_TIMESTEP]  # [3, H, W]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmaps = ["YlGnBu", "plasma", "YlOrRd"]

for i, (ax, name, cmap) in enumerate(zip(axes, channel_names, cmaps)):
    frame = np.flipud(channels[i])   # flip so origin='lower' displays north-up
    im = ax.imshow(
        frame,
        extent=[BOUNDS["left"], BOUNDS["right"], BOUNDS["bottom"], BOUNDS["top"]],
        origin="lower", cmap=cmap, aspect="auto", interpolation="bilinear",
    )
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.scatter(mapping_df["longitude"], mapping_df["latitude"],
               c="white", s=15, edgecolors="black", linewidths=0.4, zorder=5)
    visualise_singapore_outline(ax=ax)

fig.suptitle(f"CNN input channels — {timestamps[CHANNEL_TIMESTEP]}", fontsize=13)
plt.tight_layout()
plt.savefig(f"{EXPERIMENT_DIR}/analysis/cnn_input_channels.png", dpi=200, bbox_inches="tight")
plt.show()

---
# Part 3 – Grid Prediction

The CNN natively outputs a full H×W rainfall field — no re-interpolation needed.

> **Note on orientation**: the CNN grid has latitudes *descending* (top→bottom, matching
> radar convention).  We flip the H axis before calling `plot_rainfall_grid` so that
> the image is displayed north-up.

In [ ]:
# Flip H axis so rows go ascending-lat (south→north), compatible with plot_rainfall_grid
pred_fields_viz = pred_fields[:, ::-1, :].copy()  # [T, H, W]
grid_shape = (H, W)

print(f"Visualisation grid: {grid_shape}")
print(f"Predicted field range: [{pred_fields_viz.min():.3f}, {pred_fields_viz.max():.3f}] mm")

### Single Timestep

In [ ]:
TIMESTEP = 500   # ← change to any index in [0, T)

fig, ax = plt.subplots(figsize=(10, 8))
plot_rainfall_grid(
    pred_fields_viz, grid_shape, BOUNDS,
    timestamp_idx=TIMESTEP,
    mapping_df=mapping_df,
    title=f"CNN predicted rainfall — {timestamps[TIMESTEP]}",
    ax=ax,
    show_outline=True,
    **RAINFALL_SCALE,
)
visualise_with_basemap(ax=ax)
plt.tight_layout()
plt.savefig(f"{EXPERIMENT_DIR}/analysis/cnn_grid_timestep.png", dpi=200, bbox_inches="tight")
plt.show()

### Sequence of Timesteps

In [ ]:
SEQ_START = 500   # ← first timestep of the 6-panel sequence
timestep_indices = list(range(SEQ_START, SEQ_START + 6))
seq_titles = [str(timestamps[t]) for t in timestep_indices]

fig = plot_rainfall_sequence(
    pred_fields_viz, grid_shape, BOUNDS,
    timestep_indices=timestep_indices,
    mapping_df=mapping_df,
    titles=seq_titles,
    show_outline=True,
    save_path=f"{EXPERIMENT_DIR}/analysis/cnn_grid_sequence.png",
    **RAINFALL_SCALE,
)
plt.show()

### Mean Predicted Rainfall

In [ ]:
mean_pred     = pred_fields_viz.mean(axis=0)         # [H, W]
mean_pred_4d  = mean_pred[np.newaxis, :, :]           # [1, H, W]

fig, ax = plt.subplots(figsize=(10, 8))
plot_rainfall_grid(
    mean_pred_4d, grid_shape, BOUNDS,
    timestamp_idx=0,
    mapping_df=mapping_df,
    title="CNN — mean predicted rainfall over test period",
    ax=ax,
    show_outline=True,
    **RAINFALL_SCALE,
)
visualise_with_basemap(ax=ax)
plt.tight_layout()
plt.savefig(f"{EXPERIMENT_DIR}/analysis/cnn_mean_rainfall.png", dpi=200, bbox_inches="tight")
plt.show()

### Time Series Animation

In [ ]:
N_FRAMES    = 120
INTERVAL_MS = 150
ANIM_START  = 500   # ← first timestep index to animate

anim_indices    = list(range(ANIM_START, ANIM_START + N_FRAMES))
anim_timestamps = timestamps[anim_indices].tolist()

anim = animate_rainfall_grid(
    pred_fields_viz, grid_shape, BOUNDS,
    timestep_indices=anim_indices,
    timestamps=anim_timestamps,
    mapping_df=mapping_df,
    interval_ms=INTERVAL_MS,
    show_outline=True,
    save_path=f"{EXPERIMENT_DIR}/analysis/cnn_rainfall_animation.gif",
    **RAINFALL_SCALE,
)

HTML(anim.to_jshtml())